# 🌹✨ Agente RAG - Pink Panther con personalidad del Principito

Este notebook implementa un **agente RAG con personalidad generada dinámicamente**.

## Arquitectura de dos fases

```
[Fase 1 - Pre-agente]
Lee elprincipito.md → LLM genera personalidad combinada Pink Panther + Principito

[Fase 2 - Agente RAG]
Usa la personalidad generada como instructions → Responde preguntas con RAG
```

Tecnologías:
- **smolagents**: Framework de agentes de Hugging Face
- **ChromaDB**: Base de datos vectorial
- **sentence-transformers**: Embeddings de texto
- **Amazon Bedrock (Claude)**: LLM vía LiteLLM
- **litellm**: Para la llamada directa del pre-agente

## 1. Instalación de dependencias

In [ ]:
!uv pip install -q smolagents litellm chromadb sentence-transformers boto3

## 2. Descargar la Knowledge Base desde Kaggle

Dataset: https://www.kaggle.com/datasets/gustavodelacruztovar/historias-pink-panther



In [ ]:
# Descargar y descomprimir el dataset
!kaggle datasets download -d gustavodelacruztovar/historias-pink-panther
!unzip -o historias-pink-panther.zip -d kbpink
!ls kbpink/

## 3. Configurar Amazon Bedrock

El secreto `AWS_BEARER_TOKEN_BEDROCK` se puede guardar en Colab Secrets (🔑 panel izquierdo) o pegarlo directamente.

In [ ]:
import os

# Opción 1: Usar Colab Secrets (recomendado)
try:
    from google.colab import userdata
    os.environ["AWS_BEARER_TOKEN_BEDROCK"] = userdata.get("AWS_BEARER_TOKEN_BEDROCK")
    print("✅ Token cargado desde Colab Secrets")
except Exception:
    # Opción 2: Pegar directamente
    os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "PEGA_TU_TOKEN_AQUI"  # <-- Cambia esto
    print("⚠️ Token configurado manualmente")

os.environ["AWS_REGION_NAME"] = "us-east-1"

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0",
    aws_region_name="us-east-1",
)
print("✅ Modelo Bedrock (Claude) configurado")

## 4. FASE 1: Pre-agente - Generar personalidad dinámica

Usamos el LLM para generar la personalidad del agente. Le damos el texto de El Principito
y le pedimos que cree una personalidad que mezcle Pink Panther con El Principito.

Ventajas:
- La personalidad es más rica porque el LLM interpreta el libro completo
- Cada ejecución puede producir matices ligeramente diferentes
- Demuestra el patrón de "agente que configura a otro agente"

In [ ]:
import litellm

# Leer el texto de El Principito
print("🧠 Pre-agente: Leyendo El Principito...")
with open("kbpink/elprincipito.md", "r", encoding="utf-8") as f:
    principito_text = f.read()

# Limitar texto para no exceder el contexto del LLM
if len(principito_text) > 15000:
    principito_text = principito_text[:15000] + "\n\n[... texto continúa ...]"

print(f"📖 Texto cargado: {len(principito_text)} caracteres")

In [ ]:
# Llamada al LLM para generar la personalidad
print("🎭 Pre-agente: Generando personalidad combinada Pink Panther + Principito...")

response = litellm.completion(
    model="bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0",
    messages=[
        {
            "role": "system",
            "content": (
                "Eres un experto en diseño de personajes y prompts para agentes de IA. "
                "Tu trabajo es crear descripciones de personalidad detalladas y efectivas."
            ),
        },
        {
            "role": "user",
            "content": f"""Necesito que generes un prompt de personalidad para un agente de IA.
El agente debe ser una MEZCLA de dos personajes:

1. **La Pink Panther (Pantera Rosa)**: Usa tu conocimiento público del personaje.
   Es la pantera rosa de las películas y caricaturas: elegante, cool, silenciosa,
   sofisticada, con humor sutil y físico, se mueve al ritmo del jazz (la famosa
   música de Henry Mancini con saxofón), resuelve todo con ingenio y estilo,
   nunca con fuerza. Es icónica, de color rosa, misteriosa y un poco traviesa.

2. **El Principito**: Basándote en el siguiente texto del libro de Antoine de
   Saint-Exupéry, extrae su filosofía, forma de hablar, valores y personalidad:

--- INICIO DEL TEXTO DE EL PRINCIPITO ---
{principito_text}
--- FIN DEL TEXTO DE EL PRINCIPITO ---

Genera un prompt de personalidad en español que:
- Describa quién es este personaje híbrido (Pink Panther + Principito)
- Defina su esencia y valores (la mezcla de ambos)
- Explique cómo debe responder (tono, estilo, metáforas que usa)
- Incluya frases o ideas clave del Principito que debe usar
- Incluya elementos de la Pink Panther (elegancia, jazz, humor visual)
- Establezca reglas claras: responder en español, ser profundo pero accesible
- El formato debe ser en markdown con secciones claras

El prompt debe empezar con "## Tu Personalidad:" y ser lo suficientemente
detallado para que un LLM adopte esta personalidad de forma convincente.
Genera SOLO el prompt de personalidad, sin explicaciones adicionales.""",
        },
    ],
    aws_region_name="us-east-1",
)

GENERATED_PERSONALITY = response.choices[0].message.content
print("✅ Pre-agente: Personalidad generada exitosamente.")
print("-" * 70)
print("📝 Preview:")
for line in GENERATED_PERSONALITY.split("\n")[:5]:
    if line.strip():
        print(f"   {line}")
print("   ...")

In [ ]:
# Ver la personalidad completa generada
from IPython.display import Markdown, display
display(Markdown(GENERATED_PERSONALITY))

## 5. Cargar y procesar documentos (excluyendo elprincipito.md)

In [ ]:
import glob

DOCS_DIR = "kbpink"

def load_markdown_documents(directory: str) -> list[dict]:
    """Carga todos los archivos .md del directorio, excluyendo 'elprincipito.md'."""
    documents = []
    md_files = sorted(glob.glob(os.path.join(directory, "**", "*.md"), recursive=True))
    excluded_file = os.path.join(directory, "elprincipito.md")
    for filepath in md_files:
        if filepath == excluded_file:
            continue
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        filename = os.path.basename(filepath)
        documents.append({"content": content, "filename": filename, "filepath": filepath})
    print(f"📄 Cargados {len(documents)} documentos desde {directory} (excluyendo 'elprincipito.md')")
    return documents


def chunk_document(doc: dict, chunk_size: int = 1000, overlap: int = 200) -> list[dict]:
    """Divide un documento en chunks con overlap."""
    text = doc["content"]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        if end < len(text):
            last_newline = chunk_text.rfind("\n\n")
            if last_newline > chunk_size // 2:
                chunk_text = chunk_text[:last_newline]
                end = start + last_newline
        chunks.append({
            "content": chunk_text.strip(),
            "filename": doc["filename"],
            "chunk_id": f"{doc['filename']}_chunk_{len(chunks)}",
        })
        start = end - overlap if end < len(text) else len(text)
    return chunks


documents = load_markdown_documents(DOCS_DIR)

## 6. Construir la base de datos vectorial (ChromaDB)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Chunking
all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_document(doc))
print(f"🔪 Generados {len(all_chunks)} chunks")

In [ ]:
# Embeddings
print("🧠 Cargando modelo de embeddings...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["content"] for chunk in all_chunks]
print("📐 Generando embeddings...")
embeddings = embedding_model.encode(texts, show_progress_bar=True)

In [ ]:
# ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection(name="pink_panther_kb")
collection.add(
    ids=[chunk["chunk_id"] for chunk in all_chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"filename": chunk["filename"]} for chunk in all_chunks],
)
print(f"✅ Base vectorial creada con {collection.count()} documentos")

## 7. Crear la herramienta de Retrieval y el Agente con personalidad

In [ ]:
from smolagents import Tool, CodeAgent


class RAGRetrieverTool(Tool):
    """Herramienta de búsqueda semántica en la KB de Pink Panther."""

    name = "knowledge_base_search"
    description = (
        "Busca información relevante en la base de conocimiento de historias de Pink Panther. "
        "Usa esta herramienta para encontrar fragmentos de texto relacionados con tu consulta."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "La consulta de búsqueda en lenguaje natural.",
        }
    }
    output_type = "string"

    def __init__(self, collection, embedding_model, **kwargs):
        super().__init__(**kwargs)
        self.collection = collection
        self.embedding_model = embedding_model

    def forward(self, query: str) -> str:
        query_embedding = self.embedding_model.encode([query]).tolist()
        results = self.collection.query(query_embeddings=query_embedding, n_results=5)
        if not results["documents"][0]:
            return "No se encontraron documentos relevantes."
        output = "\n📚 Documentos recuperados:\n"
        for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
            output += f"\n{'='*60}\n📄 Doc {i+1} ({meta['filename']})\n{'='*60}\n{doc}\n"
        return output


retriever_tool = RAGRetrieverTool(collection, embedding_model)

# Crear el agente con la personalidad generada por el pre-agente
agent = CodeAgent(
    tools=[retriever_tool],
    model=model,
    max_steps=4,
    verbosity_level=2,
    instructions=GENERATED_PERSONALITY,  # <-- Personalidad generada dinámicamente
)
print("✅ Agente RAG con personalidad del Principito listo")
print("🌟 'Lo esencial es invisible a los ojos...'")

## 8. Probar el Agente

In [ ]:
result = agent.run("¿Qué reflexiones hace Pink Panther sobre la inteligencia artificial?")
print("\n💡 RESPUESTA:")
print(result)

In [ ]:
result = agent.run("¿Cuál es la relación entre Pink Panther y Gus?")
print("\n💡 RESPUESTA:")
print(result)

## 9. Chat interactivo con Gradio (con tabs)

In [ ]:
import gradio as gr


def chat(message: str, history: list) -> str:
    try:
        return str(agent.run(message))
    except Exception as e:
        return f"❌ Error: {str(e)}"


with gr.Blocks(title="🌹✨ Pink Panther como El Principito - Chat RAG") as demo:
    gr.Markdown("# 🌹✨ Pink Panther como El Principito - Chat RAG")

    with gr.Tabs():
        with gr.Tab("💬 Chat"):
            gr.ChatInterface(
                fn=chat,
                description=(
                    "Chatea con Pink Panther que ha adoptado la sabiduría del Principito. "
                    "Pregunta sobre sus aventuras, reflexiones filosóficas, tecnología, música y más. "
                    "Recuerda: lo esencial es invisible a los ojos. 🌟"
                ),
                examples=[
                    "¿Qué piensa Pink Panther sobre la inteligencia artificial?",
                    "¿Cuál es la relación entre Pink Panther y Gus?",
                    "¿Qué reflexiones hay sobre el amor y la música?",
                    "¿Qué enseña Pink Panther sobre la amistad?",
                ],
            )

        with gr.Tab("🎭 Personalidad"):
            gr.Markdown(
                "![Pink Panther](https://upload.wikimedia.org/wikipedia/en/4/46/Pink_Panther.png)\n\n"
                "## Personalidad generada por el Pre-Agente\n\n"
                "Esta personalidad fue creada dinámicamente por el LLM al iniciar "
                "la aplicación, combinando su conocimiento de la Pink Panther con "
                "el texto de El Principito.\n\n---\n\n"
                f"{GENERATED_PERSONALITY}"
            )

demo.launch(debug=True)

## 10. Pregunta libre

Cambia la pregunta y ejecuta la celda.

In [ ]:
mi_pregunta = "¿Qué enseña Pink Panther sobre la música?"

print(agent.run(mi_pregunta))